In [1]:
import numpy as np, pandas as pd
from cwfc.targets import load_cpu_readings, build_targets, split_train_holdout
from cwfc.models import mean_reverting_forecast
from cwfc.evaluate import collect_predictions

T = "aggregate_cpu_utilization"
cpu = load_cpu_readings("../challenge_data")
train, holdout = split_train_holdout(build_targets(cpu))   # only `train` below
assert len(train) == 110

d = collect_predictions(train, mean_reverting_forecast, T)
level = train[T].mean()

rows = []
for m in [0, 0.1, 0.2, 0.3, 0.4, 0.46, 0.5, 0.6]:
    rows.append({
        "margin_pp": m,
        "margin_pct_of_level": round(100 * m / level, 1),
        "underprov_rate": round((d["error"] > m).mean(), 3),          # actual > forecast + margin
        "avg_unused_headroom_pp": round(np.maximum(m - d["error"], 0).mean(), 3),
    })
tradeoff = pd.DataFrame(rows)
print(tradeoff)
tradeoff.to_csv("../reports/margin_tradeoff.csv", index=False)

# do the breaches come from one fold (one episode)?
print(d.assign(breach=d["error"] > 0.46).groupby("fold")["breach"].mean().round(3))

   margin_pp  margin_pct_of_level  underprov_rate  avg_unused_headroom_pp
0       0.00                  0.0           0.365                   0.107
1       0.10                  1.1           0.323                   0.172
2       0.20                  2.1           0.229                   0.244
3       0.30                  3.2           0.125                   0.326
4       0.40                  4.2           0.104                   0.414
5       0.46                  4.9           0.042                   0.470
6       0.50                  5.3           0.031                   0.508
7       0.60                  6.4           0.000                   0.608
fold
1    0.167
2    0.000
3    0.000
4    0.000
Name: breach, dtype: float64
